# Rice Grain Counting Pipeline

Notebook này triển khai pipeline đếm hạt gạo theo từng bước. Sau mỗi nhóm hàm xử lý sẽ có cell đánh giá để quan sát ảnh trung gian, mask, số component và kết quả đếm.

Pipeline chính:
1. Đọc ảnh và chuyển grayscale.
2. Khử nhiễu muối tiêu bằng median blur.
3. Hiệu chỉnh nền không đều bằng background normalization.
4. Tăng tương phản bằng CLAHE.
5. Tách foreground bằng Otsu/adaptive threshold.
6. Làm sạch mask bằng morphology và fill holes.
7. Tách hạt dính nhau bằng distance transform + watershed.
8. Lọc vùng theo đặc trưng hình học và xuất kết quả.

## 0. Cài đặt thư viện

Nếu notebook báo thiếu thư viện, chạy lệnh sau trong terminal tại root project:

```powershell
pip install -r requirements.txt
```

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import ndimage as ndi
from skimage import color, measure, morphology
from skimage.feature import peak_local_max
from skimage.segmentation import watershed

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_DIR = PROJECT_ROOT / "Dataset"
OUTPUT_DIR = PROJECT_ROOT / "output"

PARAMS = {
    "median_kernel_size": 3,
    "background_kernel_size": 51,
    "clahe_clip_limit": 2.0,
    "clahe_tile_grid_size": (8, 8),
    "morphology_kernel_size": 3,
    "adaptive_block_size": 51,
    "adaptive_c": -5,
    "min_grain_area": 50,
    "max_grain_area": 8000,
    "min_aspect_ratio": 1.2,
    "max_aspect_ratio": 10.0,
    "min_solidity": 0.45,
    "min_peak_distance": 12,
}

plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.titlesize"] = 11

print("Project root:", PROJECT_ROOT)
print("Dataset dir:", DATASET_DIR)
print("Output dir:", OUTPUT_DIR)

## 1. Hàm tiện ích hiển thị và thống kê

Các hàm này giúp đánh giá trực quan ở từng bước: hiển thị nhiều ảnh, vẽ histogram và thống kê mask.

In [ ]:
def show_images(items, cols=3):
    """Display a list of (title, image) pairs."""
    rows = int(np.ceil(len(items) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)
    for ax, (title, image) in zip(axes, items):
        if image.ndim == 2:
            ax.imshow(image, cmap="gray")
        else:
            ax.imshow(image)
        ax.set_title(title)
        ax.axis("off")
    for ax in axes[len(items):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_histogram(title, image):
    """Plot intensity distribution for a grayscale image."""
    plt.figure(figsize=(7, 3))
    plt.hist(image.ravel(), bins=64, color="steelblue")
    plt.title(title)
    plt.xlabel("Intensity")
    plt.ylabel("Pixel count")
    plt.show()


def component_count(mask):
    """Count connected components in a binary mask."""
    return int(measure.label(mask.astype(bool)).max())


def mask_summary(name, mask):
    """Return simple quality metrics for a binary mask."""
    mask_bool = mask.astype(bool)
    return {
        "step": name,
        "foreground_pixels": int(mask_bool.sum()),
        "foreground_ratio": float(mask_bool.mean()),
        "connected_components": component_count(mask_bool),
    }


def print_image_result(name, image):
    """Print basic metrics after one image-processing function."""
    arr = np.asarray(image)
    print(f"[{name}]")
    print("  shape:", arr.shape)
    print("  dtype:", arr.dtype)
    print("  min/max:", float(arr.min()), float(arr.max()))
    print("  mean/std:", float(arr.mean()), float(arr.std()))


def print_mask_result(name, mask):
    """Print binary mask metrics after one segmentation function."""
    summary = mask_summary(name, mask)
    print(f"[{name}]")
    print("  foreground pixels:", summary["foreground_pixels"])
    print("  foreground ratio:", round(summary["foreground_ratio"], 4))
    print("  connected components:", summary["connected_components"])


def print_label_result(name, labels):
    """Print label metrics after watershed or region filtering."""
    labels = np.asarray(labels)
    print(f"[{name}]")
    print("  shape:", labels.shape)
    print("  labels:", int(labels.max()))
    print("  labeled pixels:", int((labels > 0).sum()))


def make_odd(value):
    """Return the nearest valid odd integer >= 3."""
    value = int(value)
    if value % 2 == 0:
        value += 1
    return max(3, value)


def bounded_odd(value, max_value):
    """Return an odd kernel size limited by image dimensions."""
    max_value = int(max_value)
    if max_value % 2 == 0:
        max_value -= 1
    return max(3, min(make_odd(value), max_value))

## 2. Đọc ảnh và kiểm tra dataset

Hàm `read_image` đọc ảnh bằng OpenCV rồi chuyển từ BGR sang RGB để hiển thị đúng màu trong matplotlib.

In [ ]:
def list_image_files(dataset_dir):
    """List supported image files in the dataset directory."""
    extensions = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    return sorted(path for path in dataset_dir.iterdir() if path.suffix.lower() in extensions)


def read_image(path):
    """Read an image from disk as RGB."""
    image_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise FileNotFoundError(f"Cannot read image: {path}")
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

### Đánh giá hàm đọc ảnh

Cell này liệt kê ảnh trong dataset và hiển thị toàn bộ ảnh đầu vào để kiểm tra các trường hợp: ảnh bình thường, nhiễu muối tiêu, nền không đều và tương phản thấp.

In [ ]:
image_paths = list_image_files(DATASET_DIR)
print("[list_image_files]")
print(f"Found {len(image_paths)} images")
for index, path in enumerate(image_paths):
    print(index, path.name)

input_images = [(path.name, read_image(path)) for path in image_paths]
show_images(input_images, cols=2)

SAMPLE_INDEX = 0
sample_path = image_paths[SAMPLE_INDEX]
sample_image = read_image(sample_path)
print_image_result("read_image - sample image", sample_image)
print("Sample image:", sample_path.name, sample_image.shape)

## 3. Tiền xử lý ảnh

Nhóm hàm này chuẩn bị ảnh trước khi threshold:

- `to_grayscale`: chuyển ảnh sang grayscale.
- `denoise_image`: giảm nhiễu muối tiêu bằng median blur.
- `correct_illumination`: ước lượng nền bằng Gaussian blur lớn rồi chuẩn hóa ảnh theo nền.
- `enhance_contrast`: tăng tương phản cục bộ bằng CLAHE.
- `preprocess_image`: gom các bước trên và trả về ảnh trung gian.

In [ ]:
def to_grayscale(image):
    """Convert RGB or grayscale image to grayscale."""
    if image.ndim == 2:
        return image.copy()
    return cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)


def denoise_image(gray, kernel_size=3):
    """Reduce salt-and-pepper noise using median blur."""
    return cv2.medianBlur(gray, make_odd(kernel_size))


def correct_illumination(gray, kernel_size=51):
    """Correct uneven background illumination by normalizing with a blurred background."""
    max_kernel = max(3, min(gray.shape) - 1)
    kernel_size = bounded_odd(kernel_size, max_kernel)
    background = cv2.GaussianBlur(gray, (kernel_size, kernel_size), 0)
    corrected = gray.astype(np.float32) / (background.astype(np.float32) + 1.0)
    corrected *= np.mean(background)
    corrected = np.clip(corrected, 0, 255).astype(np.uint8)
    return corrected, background


def enhance_contrast(gray, clip_limit=2.0, tile_grid_size=(8, 8)):
    """Improve local contrast using CLAHE."""
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    return clahe.apply(gray)


def preprocess_image(image, params):
    """Run all preprocessing steps and return intermediate images."""
    gray = to_grayscale(image)
    denoised = denoise_image(gray, params["median_kernel_size"])
    corrected, background = correct_illumination(denoised, params["background_kernel_size"])
    enhanced = enhance_contrast(
        corrected,
        params["clahe_clip_limit"],
        params["clahe_tile_grid_size"],
    )
    return {
        "gray": gray,
        "denoised": denoised,
        "background": background,
        "corrected": corrected,
        "enhanced": enhanced,
    }

### Đánh giá nhóm hàm tiền xử lý

Quan sát ảnh sau từng bước. Với ảnh nhiễu muối tiêu, ảnh `denoised` phải ít điểm nhiễu hơn. Với ảnh nền không đều, ảnh `corrected` phải giảm ảnh hưởng sáng tối dạng sóng. Với ảnh tương phản thấp, ảnh `enhanced` phải làm hạt rõ hơn.

In [ ]:
print_image_result("input image", sample_image)

gray = to_grayscale(sample_image)
print_image_result("to_grayscale", gray)

denoised = denoise_image(gray, PARAMS["median_kernel_size"])
print_image_result("denoise_image", denoised)

corrected, background = correct_illumination(denoised, PARAMS["background_kernel_size"])
print_image_result("correct_illumination - background", background)
print_image_result("correct_illumination - corrected", corrected)

enhanced = enhance_contrast(
    corrected,
    PARAMS["clahe_clip_limit"],
    PARAMS["clahe_tile_grid_size"],
)
print_image_result("enhance_contrast", enhanced)

pre = {
    "gray": gray,
    "denoised": denoised,
    "background": background,
    "corrected": corrected,
    "enhanced": enhanced,
}

show_images([
    ("original", sample_image),
    ("gray", gray),
    ("denoised", denoised),
    ("estimated background", background),
    ("illumination corrected", corrected),
    ("CLAHE enhanced", enhanced),
], cols=3)

show_histogram("Histogram - grayscale", pre["gray"])
show_histogram("Histogram - enhanced", pre["enhanced"])

## 4. Phân đoạn hạt gạo

Nhóm hàm này tạo mask nhị phân:

- `threshold_grains`: dùng Otsu trước; nếu tỷ lệ foreground bất thường thì chuyển sang adaptive threshold.
- `clean_mask`: xóa vùng nhỏ, opening/closing và fill holes.
- `segment_grains`: trả về mask thô, mask đã làm sạch và thống kê.

In [ ]:
def adaptive_threshold(gray, block_size=51, c=-5):
    """Adaptive threshold for images with uneven background."""
    block_size = make_odd(block_size)
    return cv2.adaptiveThreshold(
        gray,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        block_size,
        c,
    ).astype(bool)


def threshold_grains(enhanced, params):
    """Create an initial grain mask using Otsu with adaptive fallback."""
    threshold_value, otsu_mask = cv2.threshold(
        enhanced,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU,
    )
    mask = otsu_mask.astype(bool)
    foreground_ratio = float(mask.mean())
    method = "otsu"

    if foreground_ratio < 0.005 or foreground_ratio > 0.60:
        mask = adaptive_threshold(
            enhanced,
            params["adaptive_block_size"],
            params["adaptive_c"],
        )
        method = "adaptive"

    info = {
        "threshold_value": float(threshold_value),
        "method": method,
        "foreground_ratio": float(mask.mean()),
    }
    return mask, info


def clean_mask(mask, params):
    """Remove noise and repair grain regions in a binary mask."""
    min_area = params["min_grain_area"]
    kernel_size = make_odd(params["morphology_kernel_size"])
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))

    no_small = morphology.remove_small_objects(mask.astype(bool), min_size=min_area)
    opened = cv2.morphologyEx((no_small.astype(np.uint8) * 255), cv2.MORPH_OPEN, kernel) > 0
    closed = cv2.morphologyEx((opened.astype(np.uint8) * 255), cv2.MORPH_CLOSE, kernel) > 0
    filled = ndi.binary_fill_holes(closed)
    cleaned = morphology.remove_small_objects(filled.astype(bool), min_size=min_area)

    return cleaned


def segment_grains(enhanced, params):
    """Return raw mask, cleaned mask, and segmentation metrics."""
    raw_mask, threshold_info = threshold_grains(enhanced, params)
    cleaned_mask = clean_mask(raw_mask, params)
    metrics = [
        mask_summary("raw_threshold", raw_mask),
        mask_summary("cleaned_mask", cleaned_mask),
    ]
    metrics[0].update(threshold_info)
    return raw_mask, cleaned_mask, pd.DataFrame(metrics)

### Đánh giá nhóm hàm phân đoạn

Ảnh `raw mask` cho biết threshold ban đầu. Ảnh `cleaned mask` phải bớt nhiễu nhỏ, vùng hạt đầy hơn và số component hợp lý hơn.

In [ ]:
raw_mask, threshold_info = threshold_grains(pre["enhanced"], PARAMS)
print("[threshold_grains]")
display(pd.DataFrame([threshold_info]))
print_mask_result("threshold_grains - raw mask", raw_mask)

cleaned_mask = clean_mask(raw_mask, PARAMS)
print_mask_result("clean_mask", cleaned_mask)

segmentation_metrics = pd.DataFrame([
    {**mask_summary("raw_threshold", raw_mask), **threshold_info},
    mask_summary("cleaned_mask", cleaned_mask),
])

show_images([
    ("enhanced", pre["enhanced"]),
    ("raw mask", raw_mask),
    ("cleaned mask", cleaned_mask),
], cols=3)

display(segmentation_metrics)

## 5. Tách hạt dính nhau và đếm

Nhóm hàm này dùng distance transform và watershed:

- `watershed_separation`: tìm marker từ local maxima trên distance map rồi watershed.
- `filter_grain_regions`: loại vùng không giống hạt gạo theo diện tích, aspect ratio và solidity.
- `count_grains`: trả về label cuối cùng và bảng đặc trưng từng hạt.

In [ ]:
def watershed_separation(mask, params):
    """Separate touching grains using distance transform and watershed."""
    distance = ndi.distance_transform_edt(mask)
    coords = peak_local_max(
        distance,
        min_distance=params["min_peak_distance"],
        labels=mask,
    )

    markers = np.zeros(mask.shape, dtype=np.int32)
    for marker_id, (row, col) in enumerate(coords, start=1):
        markers[row, col] = marker_id

    if markers.max() == 0:
        markers = measure.label(mask.astype(bool)).astype(np.int32)

    labels = watershed(-distance, markers, mask=mask.astype(bool))
    return labels, distance, markers


def filter_grain_regions(labels, params):
    """Filter watershed regions by grain-like shape constraints."""
    filtered = np.zeros(labels.shape, dtype=np.int32)
    rows = []
    next_id = 1

    for region in measure.regionprops(labels):
        min_row, min_col, max_row, max_col = region.bbox
        height = max_row - min_row
        width = max_col - min_col
        short_side = max(1, min(height, width))
        long_side = max(height, width)
        aspect_ratio = long_side / short_side

        accepted = (
            params["min_grain_area"] <= region.area <= params["max_grain_area"]
            and params["min_aspect_ratio"] <= aspect_ratio <= params["max_aspect_ratio"]
            and region.solidity >= params["min_solidity"]
        )

        rows.append({
            "label": int(region.label),
            "area": int(region.area),
            "aspect_ratio": float(aspect_ratio),
            "solidity": float(region.solidity),
            "accepted": bool(accepted),
        })

        if accepted:
            filtered[labels == region.label] = next_id
            next_id += 1

    return filtered, pd.DataFrame(rows)


def count_grains(mask, params):
    """Run watershed separation, region filtering, and return count outputs."""
    labels, distance, markers = watershed_separation(mask, params)
    filtered_labels, region_table = filter_grain_regions(labels, params)
    count = int(filtered_labels.max())
    rejected = int((~region_table["accepted"]).sum()) if len(region_table) else 0
    return {
        "labels": labels,
        "filtered_labels": filtered_labels,
        "distance": distance,
        "markers": markers,
        "regions": region_table,
        "count": count,
        "rejected_regions": rejected,
    }

### Đánh giá nhóm hàm watershed và counting

Kiểm tra distance map, marker, watershed label và label sau khi lọc. Bảng region cho biết vùng nào được chấp nhận hoặc bị loại.

In [ ]:
labels, distance, markers = watershed_separation(cleaned_mask, PARAMS)
print_image_result("watershed_separation - distance", distance)
print_label_result("watershed_separation - markers", markers)
print_label_result("watershed_separation - labels", labels)

filtered_labels, region_table = filter_grain_regions(labels, PARAMS)
print_label_result("filter_grain_regions - filtered labels", filtered_labels)
print("[filter_grain_regions]")
print("  total regions:", len(region_table))
print("  accepted regions:", int(region_table["accepted"].sum()) if len(region_table) else 0)
print("  rejected regions:", int((~region_table["accepted"]).sum()) if len(region_table) else 0)

count_output = count_grains(cleaned_mask, PARAMS)
print("[count_grains]")
print("  final count:", count_output["count"])
print("  rejected regions:", count_output["rejected_regions"])

label_rgb = color.label2rgb(count_output["labels"], bg_label=0)
filtered_rgb = color.label2rgb(count_output["filtered_labels"], bg_label=0)

show_images([
    ("cleaned mask", cleaned_mask),
    ("distance transform", count_output["distance"]),
    ("markers", count_output["markers"]),
    ("watershed labels", label_rgb),
    ("filtered labels", filtered_rgb),
], cols=3)

display(count_output["regions"].head(20))

## 6. Lưu kết quả và chạy toàn bộ pipeline

Các hàm dưới đây lưu mask, label, contour overlay và file CSV tổng hợp. Đây là phần dùng để tạo output cuối cùng cho báo cáo.

In [ ]:
def ensure_output_dirs(output_dir):
    """Create output directories used by the notebook."""
    for child in ["masks", "contours", "labels", "intermediate"]:
        (output_dir / child).mkdir(parents=True, exist_ok=True)


def labels_to_uint8(labels):
    """Convert integer labels to a visible uint8 image."""
    if labels.max() == 0:
        return np.zeros(labels.shape, dtype=np.uint8)
    return ((labels.astype(np.float32) / labels.max()) * 255).astype(np.uint8)


def contour_overlay(original, labels):
    """Draw contours from labels on top of the original image."""
    overlay = original.copy()
    binary = (labels > 0).astype(np.uint8) * 255
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(overlay, contours, -1, (255, 0, 0), 2)
    return overlay


def save_outputs(image_path, original, pre, raw_mask, cleaned_mask, count_output, output_dir):
    """Save intermediate and final images for one input image."""
    ensure_output_dirs(output_dir)
    stem = image_path.stem

    cv2.imwrite(str(output_dir / "masks" / f"{stem}_raw_mask.png"), raw_mask.astype(np.uint8) * 255)
    cv2.imwrite(str(output_dir / "masks" / f"{stem}_cleaned_mask.png"), cleaned_mask.astype(np.uint8) * 255)
    cv2.imwrite(str(output_dir / "labels" / f"{stem}_labels.png"), labels_to_uint8(count_output["filtered_labels"]))

    overlay = contour_overlay(original, count_output["filtered_labels"])
    cv2.imwrite(
        str(output_dir / "contours" / f"{stem}_contours.png"),
        cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR),
    )

    cv2.imwrite(str(output_dir / "intermediate" / f"{stem}_gray.png"), pre["gray"])
    cv2.imwrite(str(output_dir / "intermediate" / f"{stem}_corrected.png"), pre["corrected"])
    cv2.imwrite(str(output_dir / "intermediate" / f"{stem}_enhanced.png"), pre["enhanced"])


def run_one_image(image_path, params, output_dir=None):
    """Run full rice counting pipeline for one image."""
    original = read_image(image_path)
    pre = preprocess_image(original, params)
    raw_mask, cleaned_mask, segmentation_metrics = segment_grains(pre["enhanced"], params)
    count_output = count_grains(cleaned_mask, params)

    if output_dir is not None:
        save_outputs(image_path, original, pre, raw_mask, cleaned_mask, count_output, output_dir)

    result = {
        "image": image_path.name,
        "count": count_output["count"],
        "rejected_regions": count_output["rejected_regions"],
        "threshold_method": segmentation_metrics.loc[0, "method"],
        "foreground_ratio": segmentation_metrics.loc[1, "foreground_ratio"],
        "components_after_cleaning": segmentation_metrics.loc[1, "connected_components"],
    }
    return result, original, pre, raw_mask, cleaned_mask, count_output

### Đánh giá pipeline trên toàn bộ dataset

Cell này chạy tất cả ảnh trong `Dataset`, lưu output và tạo bảng `output/results.csv`.

In [ ]:
ensure_output_dirs(OUTPUT_DIR)

batch_results = []
last_debug = None
for path in image_paths:
    result, original, pre, raw_mask, cleaned_mask, count_output = run_one_image(path, PARAMS, OUTPUT_DIR)
    print("[run_one_image]", result)
    batch_results.append(result)
    last_debug = (path, original, pre, raw_mask, cleaned_mask, count_output)

results_df = pd.DataFrame(batch_results)
results_csv = OUTPUT_DIR / "results.csv"
results_df.to_csv(results_csv, index=False, encoding="utf-8-sig")

display(results_df)
print("Saved:", results_csv)

### Kiểm tra trực quan kết quả cuối

Cell này hiển thị contour overlay cho từng ảnh. Nếu thấy đếm thiếu hoặc đếm dư, điều chỉnh các tham số trong `PARAMS`, thường là `min_grain_area`, `min_peak_distance`, `min_aspect_ratio` và `min_solidity`.

In [ ]:
overlays = []
for path in image_paths:
    result, original, pre, raw_mask, cleaned_mask, count_output = run_one_image(path, PARAMS, None)
    overlay = contour_overlay(original, count_output["filtered_labels"])
    overlays.append((f"{path.name} | count={result['count']}", overlay))

show_images(overlays, cols=2)

## 7. Nhận xét khi báo cáo

Khi viết báo cáo, nên đánh giá từng loại ảnh:

- Ảnh bình thường: kiểm tra số contour có khớp trực quan không.
- Ảnh nhiễu muối tiêu: so sánh trước/sau median blur và số component sau cleaning.
- Ảnh nền không đều: kiểm tra ảnh background và ảnh corrected.
- Ảnh tương phản thấp: kiểm tra histogram trước/sau CLAHE.
- Hạt dính nhau: kiểm tra marker và watershed label.

Nếu cần chỉnh tham số, chỉnh trong `PARAMS` rồi chạy lại từ phần tiền xử lý.